# Feature Engineering

The objective of feature engineering is to transform the cleaned dataset into informative features that can help machine learning models capture important patterns in credit default risk.

The features created in this stage are motivated by the findings from Business EDA, particularly:

- Previous payment delinquency
- Credit utilization
- Customer age
- Income
- Debt and credit exposure

Feature engineering will be performed carefully to avoid unnecessary complexity and data leakage.

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("../Data/Processed/credit_cleaned.csv")

# Create a copy for analysis
credit_clean = df.copy()

In [3]:
credit_clean.head()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0
1,0,0.957151,40,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0
2,0,0.658180,38,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0
3,0,0.233810,30,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0
4,0,0.907239,49,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0


In [4]:
fe_df = credit_clean.copy()

In [5]:
fe_df.shape

(149999, 11)

## 1. Total Delinquency

### Objective

Create a combined measure representing the total number of recorded delinquency events across the 30–59 day, 60–89 day, and 90+ day categories.

This feature is motivated by the Business EDA finding that previous payment delinquency is strongly associated with future default.

In [6]:
fe_df["TotalDelinquency"] = (
    fe_df["NumberOfTime30-59DaysPastDueNotWorse"]
    + fe_df["NumberOfTime60-89DaysPastDueNotWorse"]
    + fe_df["NumberOfTimes90DaysLate"]
)


In [7]:
fe_df["TotalDelinquency"].describe()

count    149999.000000
mean          0.400349
std           1.101292
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          19.000000
Name: TotalDelinquency, dtype: float64

In [9]:
fe_df["TotalDelinquency"].value_counts().sort_index().head(15)

TotalDelinquency
0.0     119906
1.0      17242
2.0       5942
3.0       2884
4.0       1595
5.0        940
6.0        592
7.0        380
8.0        197
9.0        125
10.0        73
11.0        53
12.0        18
13.0        19
14.0        13
Name: count, dtype: int64

In [10]:
fe_df[
    [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
        "TotalDelinquency"
    ]
].head(10)

,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,TotalDelinquency
0,2.0,0.0,0.0,2.0
1,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,2.0
3,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,1.0
5,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0


## 2. Previous Delinquency Indicator

### Objective

Create a binary feature indicating whether a customer has experienced at least one previous delinquency.

This captures the distinction between customers with no delinquency history and those with any history of payment problems.

In [11]:
fe_df["HasDelinquency"] = (
    fe_df["TotalDelinquency"] > 0
).astype(int)

In [12]:
fe_df["HasDelinquency"].value_counts()

HasDelinquency
0    119906
1     30093
Name: count, dtype: int64

In [13]:
fe_df["HasDelinquency"].value_counts(normalize=True) * 100

HasDelinquency
0    79.937866
1    20.062134
Name: proportion, dtype: float64

## 3. Severe Delinquency Indicator

### Objective

Create a binary feature indicating whether a customer has previously experienced at least one 90+ day late payment.

90+ day delinquency represents a more severe repayment problem and showed a particularly strong association with future default during Business EDA.

In [14]:
fe_df["HasSevereDelinquency"] = (
    fe_df["NumberOfTimes90DaysLate"] > 0
).astype(int)

In [16]:
fe_df["HasSevereDelinquency"].value_counts()

HasSevereDelinquency
0    141930
1      8069
Name: count, dtype: int64

In [17]:
fe_df["HasSevereDelinquency"].value_counts(normalize=True) * 100

HasSevereDelinquency
0    94.620631
1     5.379369
Name: proportion, dtype: float64

## 4. Log-Transformed Credit Utilization

### Objective

Create a logarithmically transformed version of revolving credit utilization.

The original feature is extremely right-skewed, with most observations concentrated at relatively low values and a small number of extreme observations. Applying `log1p` compresses the extreme values while preserving zero-valued observations.

The original feature is retained, allowing model comparison between the raw and transformed representations.

In [19]:
fe_df["LogCreditUtilization"] = np.log1p(
    fe_df["RevolvingUtilizationOfUnsecuredLines"]
)

In [20]:
fe_df[
    [
        "RevolvingUtilizationOfUnsecuredLines",
        "LogCreditUtilization"
    ]
].describe()

,RevolvingUtilizationOfUnsecuredLines,LogCreditUtilization
count,149999.000000,149999.000000
mean,6.048472,0.257897
std,249.756203,0.384852
min,0.000000,0.000000
25%,0.029867,0.029430
50%,0.154176,0.143386
75%,0.559044,0.444073
max,50708.000000,10.833859


## 5. High Credit Utilization Indicator

### Objective

Create a binary indicator for customers whose revolving credit utilization exceeds 1.

Business EDA showed a substantial increase in default rates around and above this level.

In [21]:
fe_df["HighCreditUtilization"] = (
    fe_df["RevolvingUtilizationOfUnsecuredLines"] > 1
).astype(int)

In [22]:
fe_df["HighCreditUtilization"].value_counts()

HighCreditUtilization
0    146678
1      3321
Name: count, dtype: int64

## 6. Log-Transformed Monthly Income

### Objective

Create a logarithmically transformed version of MonthlyIncome to reduce the effect of extreme income values and provide a more balanced representation of income differences.

The original MonthlyIncome feature is retained so that the model can potentially use either representation.

In [23]:
fe_df["LogMonthlyIncome"] = np.log1p(
    fe_df["MonthlyIncome"]
)

In [25]:
fe_df[
    [
        "MonthlyIncome",
        "LogMonthlyIncome"
    ]
].describe()

,MonthlyIncome,LogMonthlyIncome
count,1.499990e+05,149999.000000
mean,6.418458e+03,8.447403
std,1.289044e+04,1.195800
min,0.000000e+00,0.000000
25%,3.903000e+03,8.269757
50%,5.400000e+03,8.594339
75%,7.400000e+03,8.909370
max,3.008750e+06,14.917036


In [26]:
fe_df[
    [
        "MonthlyIncome",
        "LogMonthlyIncome"
    ]
].head(10)

,MonthlyIncome,LogMonthlyIncome
0,9120.0,9.118335
1,2600.0,7.863651
2,3042.0,8.020599
3,3300.0,8.101981
4,63588.0,11.060196
5,3500.0,8.160804
6,5400.0,8.594339
7,3500.0,8.160804
8,5400.0,8.594339
9,23684.0,10.072597


## 7. Income Category

### Objective

Create an income category based on the distribution of MonthlyIncome.

Business EDA showed that customers in lower income ranges have higher default rates. Categorizing income can provide a simple representation of this pattern.

In [27]:
fe_df["IncomeCategory"] = pd.cut(
    fe_df["MonthlyIncome"],
    bins=[-1, 3400, 5400, 8250, float("inf")],
    labels=[
        "Low",
        "Lower-Middle",
        "Upper-Middle",
        "High"
    ]
)

In [28]:
fe_df["IncomeCategory"].value_counts()

IncomeCategory
Lower-Middle    59757
Low             30289
High            29993
Upper-Middle    29960
Name: count, dtype: int64

## 8. Log-Transformed Debt Ratio

### Objective

Create a logarithmically transformed version of DebtRatio to reduce the influence of its extreme right-skewed distribution.

The original DebtRatio contains extremely large values, while most observations are concentrated at much smaller values. The `log1p` transformation compresses these extreme values and provides a more balanced representation for models that may be sensitive to feature scale and skewness.

The original DebtRatio is retained for comparison.

In [29]:
fe_df["LogDebtRatio"] = np.log1p(
    fe_df["DebtRatio"]
)

In [30]:
fe_df[
    [
        "DebtRatio",
        "LogDebtRatio"
    ]
].describe()

,DebtRatio,LogDebtRatio
count,149999.000000,149999.000000
mean,353.007426,1.525182
std,2037.825113,2.627244
min,0.000000,0.000000
25%,0.175074,0.161331
50%,0.366503,0.312255
75%,0.868257,0.625006
max,329664.000000,12.705832


In [31]:
fe_df[
    [
        "DebtRatio",
        "LogDebtRatio"
    ]
].head(10)

,DebtRatio,LogDebtRatio
0,0.802982,0.589442
1,0.121876,0.115002
2,0.085113,0.081684
3,0.036050,0.035415
4,0.024926,0.024620
5,0.375607,0.318895
6,5710.000000,8.650149
7,0.209940,0.190571
8,46.000000,3.850148
9,0.606291,0.473928


## 9. High Debt Ratio Indicator

### Objective

Create a binary indicator identifying customers whose DebtRatio exceeds 1.

This provides the model with an explicit representation of customers whose debt-to-income ratio is above the normal 1.0 threshold.

In [32]:
fe_df["HighDebtRatio"] = (
    fe_df["DebtRatio"] > 1
).astype(int)

In [33]:
fe_df["HighDebtRatio"].value_counts()

HighDebtRatio
0    114862
1     35137
Name: count, dtype: int64

In [34]:
fe_df["HighDebtRatio"].value_counts(normalize=True) * 100

HighDebtRatio
0    76.575177
1    23.424823
Name: proportion, dtype: float64

## 10. Total Credit Accounts

### Objective

Create a combined measure of the customer's open credit lines and real-estate-related loans.

This provides an overall measure of the customer's active credit exposure using the available account-count variables.

In [35]:
fe_df["TotalCreditAccounts"] = (
    fe_df["NumberOfOpenCreditLinesAndLoans"]
    + fe_df["NumberRealEstateLoansOrLines"]
)

In [36]:
fe_df["TotalCreditAccounts"].describe()

count    149999.000000
mean          9.471010
std           5.727412
min           0.000000
25%           5.000000
50%           9.000000
75%          13.000000
max         112.000000
Name: TotalCreditAccounts, dtype: float64

## FINAL REVIEW

In [37]:
fe_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149999 entries, 0 to 149998
Data columns (total 21 columns):
 #   Column                                Non-Null Count   Dtype   
---  ------                                --------------   -----   
 0   SeriousDlqin2yrs                      149999 non-null  int64   
 1   RevolvingUtilizationOfUnsecuredLines  149999 non-null  float64 
 2   age                                   149999 non-null  int64   
 3   NumberOfTime30-59DaysPastDueNotWorse  149999 non-null  float64 
 4   DebtRatio                             149999 non-null  float64 
 5   MonthlyIncome                         149999 non-null  float64 
 6   NumberOfOpenCreditLinesAndLoans       149999 non-null  int64   
 7   NumberOfTimes90DaysLate               149999 non-null  float64 
 8   NumberRealEstateLoansOrLines          149999 non-null  int64   
 9   NumberOfTime60-89DaysPastDueNotWorse  149999 non-null  float64 
 10  NumberOfDependents                    149999 non-null  float64 
 11

In [38]:
fe_df.isnull().sum()

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
TotalDelinquency                        0
HasDelinquency                          0
HasSevereDelinquency                    0
LogCreditUtilization                    0
HighCreditUtilization                   0
LogMonthlyIncome                        0
IncomeCategory                          0
LogDebtRatio                            0
HighDebtRatio                           0
TotalCreditAccounts                     0
dtype: int64

In [39]:
fe_df.columns.tolist()

['SeriousDlqin2yrs',
 'RevolvingUtilizationOfUnsecuredLines',
 'age',
 'NumberOfTime30-59DaysPastDueNotWorse',
 'DebtRatio',
 'MonthlyIncome',
 'NumberOfOpenCreditLinesAndLoans',
 'NumberOfTimes90DaysLate',
 'NumberRealEstateLoansOrLines',
 'NumberOfTime60-89DaysPastDueNotWorse',
 'NumberOfDependents',
 'TotalDelinquency',
 'HasDelinquency',
 'HasSevereDelinquency',
 'LogCreditUtilization',
 'HighCreditUtilization',
 'LogMonthlyIncome',
 'IncomeCategory',
 'LogDebtRatio',
 'HighDebtRatio',
 'TotalCreditAccounts']

In [40]:
np.isinf(
    fe_df.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

In [41]:
engineered_features = [
    "TotalDelinquency",
    "HasDelinquency",
    "HasSevereDelinquency",
    "LogCreditUtilization",
    "HighCreditUtilization",
    "LogMonthlyIncome",
    "IncomeCategory",
    "LogDebtRatio",
    "HighDebtRatio",
    "TotalCreditAccounts"
]

fe_df[engineered_features].describe(include="all")

,TotalDelinquency,HasDelinquency,HasSevereDelinquency,LogCreditUtilization,HighCreditUtilization,LogMonthlyIncome,IncomeCategory,LogDebtRatio,HighDebtRatio,TotalCreditAccounts
count,149999.000000,149999.000000,149999.000000,149999.000000,149999.00000,149999.000000,149999,149999.000000,149999.000000,149999.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,Lower-Middle,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,59757,NaN,NaN,NaN
mean,0.400349,0.200621,0.053794,0.257897,0.02214,8.447403,NaN,1.525182,0.234248,9.471010
std,1.101292,0.400467,0.225611,0.384852,0.14714,1.195800,NaN,2.627244,0.423529,5.727412
min,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,NaN,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.029430,0.00000,8.269757,NaN,0.161331,0.000000,5.000000
50%,0.000000,0.000000,0.000000,0.143386,0.00000,8.594339,NaN,0.312255,0.000000,9.000000
75%,0.000000,0.000000,0.000000,0.444073,0.00000,8.909370,NaN,0.625006,0.000000,13.000000


In [42]:
numeric_df = fe_df.select_dtypes(include=np.number)

corr = numeric_df.corr()

corr["SeriousDlqin2yrs"].sort_values(ascending=False)

SeriousDlqin2yrs                        1.000000
TotalDelinquency                        0.388673
HasSevereDelinquency                    0.329598
NumberOfTimes90DaysLate                 0.311713
HasDelinquency                          0.306863
NumberOfTime30-59DaysPastDueNotWorse    0.271414
NumberOfTime60-89DaysPastDueNotWorse    0.265594
HighCreditUtilization                   0.184146
LogCreditUtilization                    0.178770
NumberOfDependents                      0.046871
RevolvingUtilizationOfUnsecuredLines   -0.001802
HighDebtRatio                          -0.003629
NumberRealEstateLoansOrLines           -0.007037
DebtRatio                              -0.007602
MonthlyIncome                          -0.017151
LogDebtRatio                           -0.017195
LogMonthlyIncome                       -0.017617
TotalCreditAccounts                    -0.028045
NumberOfOpenCreditLinesAndLoans        -0.029669
age                                    -0.115397
Name: SeriousDlqin2y

In [43]:
fe_df.duplicated().sum()

np.int64(796)

In [45]:
target = "SeriousDlqin2yrs"

In [46]:
X = fe_df.drop(columns=[target])
y = fe_df[target]

In [47]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (149999, 20)
y shape: (149999,)


In [48]:
y.value_counts(normalize=True) * 100

SeriousDlqin2yrs
0    93.315955
1     6.684045
Name: proportion, dtype: float64

In [49]:
X.dtypes

RevolvingUtilizationOfUnsecuredLines     float64
age                                        int64
NumberOfTime30-59DaysPastDueNotWorse     float64
DebtRatio                                float64
MonthlyIncome                            float64
NumberOfOpenCreditLinesAndLoans            int64
NumberOfTimes90DaysLate                  float64
NumberRealEstateLoansOrLines               int64
NumberOfTime60-89DaysPastDueNotWorse     float64
NumberOfDependents                       float64
TotalDelinquency                         float64
HasDelinquency                             int64
HasSevereDelinquency                       int64
LogCreditUtilization                     float64
HighCreditUtilization                      int64
LogMonthlyIncome                         float64
IncomeCategory                          category
LogDebtRatio                             float64
HighDebtRatio                              int64
TotalCreditAccounts                        int64
dtype: object

# Feature Engineering Summary

The feature engineering stage created a small set of business-motivated features based on the findings from Business EDA.

### Engineered Features

| Feature | Purpose |
|---|---|
| `TotalDelinquency` | Captures overall delinquency frequency |
| `HasDelinquency` | Indicates whether any previous delinquency occurred |
| `HasSevereDelinquency` | Indicates previous 90+ day delinquency |
| `LogCreditUtilization` | Reduces extreme right skew in credit utilization |
| `HighCreditUtilization` | Identifies utilization above 1 |
| `LogMonthlyIncome` | Reduces extreme right skew in income |
| `IncomeCategory` | Represents broad income levels |
| `LogDebtRatio` | Reduces extreme right skew in DebtRatio |
| `HighDebtRatio` | Identifies DebtRatio values above 1 |
| `TotalCreditAccounts` | Represents overall credit account exposure |

The engineered features are based on observed business patterns and domain reasoning rather than arbitrary feature creation.

The target variable `SeriousDlqin2yrs` is kept separate from the feature matrix.

The resulting dataset will now be used for model development and evaluation.

In [51]:
fe_df.to_csv(
    "../Data/Processed/credit_feature_engineered.csv",
    index=False
)